# Qwen2.5-Coder-14B-Instruct → Q4_K_M GGUF

This notebook downloads **Qwen2.5-Coder-14B-Instruct** from HuggingFace, converts it to GGUF (F16), then quantizes it to Q4_K_M using llama.cpp.

**Why this model?**
- Purpose-built for coding, trained on 5.5 trillion tokens of source code across all major languages
- 14B parameters — large enough to be highly capable, small enough to convert on a T4 GPU
- Q4_K_M output is ~9 GB — straightforward pipeline with no architecture quirks

**Estimated time on Colab T4:**
- Download: ~10–15 min
- Conversion to F16: ~10 min
- Quantization to Q4_K_M: ~20–25 min

> ⚠️ Make sure you have **High-RAM** enabled in Colab (`Runtime → Change runtime type`) as the F16 intermediate file is ~29 GB and requires enough disk + RAM to process.

## Step 1: Install system dependencies and clone llama.cpp

In [ ]:
!apt-get install -y build-essential cmake git
!pip install huggingface_hub
!git clone --depth=1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

## Step 2: Install Python dependencies

In [ ]:
!pip install torch transformers sentencepiece protobuf numpy gguf

## Step 3: Download Qwen2.5-Coder-14B-Instruct from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
import os

# If the model is gated, uncomment and set your HF token:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

model_id = "Qwen/Qwen2.5-Coder-14B-Instruct"
local_dir = "/content/qwen2.5-coder-14b-instruct"

snapshot_download(
    repo_id=model_id,
    local_dir=local_dir,
    ignore_patterns=["*.bin", "*.pt"]  # download safetensors only
)
print("Download complete!")

## Step 4: Convert HuggingFace model to GGUF (F16)

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/qwen2.5-coder-14b-instruct \
    --outfile /content/qwen2.5-coder-14b-instruct-f16.gguf \
    --outtype f16

## Step 5: Download pre-built llama.cpp binaries (latest release)

In [ ]:
import requests

# Get the latest release tag from GitHub
release = requests.get("https://api.github.com/repos/ggerganov/llama.cpp/releases/latest").json()
tag = release["tag_name"]
print(f"Latest llama.cpp release: {tag}")

# Download the Ubuntu x64 tarball
url = f"https://github.com/ggerganov/llama.cpp/releases/download/{tag}/llama-{tag}-bin-ubuntu-x64.tar.gz"
print(f"Downloading: {url}")
!wget -q "{url}" -O /content/llama.tar.gz

# Extract
!mkdir -p /content/llama-bin
!tar -xzf /content/llama.tar.gz -C /content/llama-bin

# Find the quantize binary
!find /content/llama-bin -name "llama-quantize"

## Step 6: Quantize to Q4_K_M

In [ ]:
import glob

# Auto-detect the quantize binary path
matches = glob.glob("/content/llama-bin/**/llama-quantize", recursive=True)
if not matches:
    raise FileNotFoundError("llama-quantize not found! Check the extraction in Step 5.")
quantize_bin = matches[0]
print(f"Using: {quantize_bin}")

!chmod +x "{quantize_bin}"
!"{quantize_bin}" \
    /content/qwen2.5-coder-14b-instruct-f16.gguf \
    /content/qwen2.5-coder-14b-instruct-Q4_K_M.gguf \
    Q4_K_M

## Step 7: Save to Google Drive

Mount your Drive and copy the quantized file there. Strongly recommended over direct download for a ~9 GB file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/qwen2.5-coder-14b-instruct-Q4_K_M.gguf "/content/drive/MyDrive/qwen2.5-coder-14b-instruct-Q4_K_M.gguf"
print("Done! File saved to Google Drive.")

## (Optional) Step 8: Direct download to your computer

Only use this if you don't have Drive access. Note: downloading ~9 GB directly from Colab is slow and likely to time out — Drive is strongly preferred.

In [ ]:
from google.colab import files
files.download("/content/qwen2.5-coder-14b-instruct-Q4_K_M.gguf")